In [1]:
import requests
import pandas as pd
import random
import time
import os
from datetime import datetime, timedelta

In [2]:
# APIのエンドポイント
NAROU_API_URL = "https://api.syosetu.com/novelapi/api/"

# APIへの負荷を考慮したリクエスト待機時間（秒）
REQUEST_INTERVAL = 0.1

# 小説家になろうの最大ユーザID（2025年時点の情報を参考に設定）
MAX_USER_ID = 2814253

# --- 保存先ディレクトリとファイルパス ---
# このノートブックファイルがある場所からの相対パスで指定
DATASET_DIR = "dataset"
RAW_DATA_PATH = os.path.join(DATASET_DIR, "narou_dataset_raw.csv")
PROCESSED_DATA_PATH = os.path.join(DATASET_DIR, "narou_dataset.csv")

# --- 進捗とエラーを記録するファイルパス ---
PROGRESS_FILE_PATH = os.path.join(DATASET_DIR, "progress_uid.txt")
ERROR_LOG_PATH = os.path.join(DATASET_DIR, "error_log.txt")

In [3]:
def get_narou_data_by_userid(user_id):
    """
    指定されたユーザIDのすべての小説データを取得します。
    (1ユーザあたり最大500件まで)
    """
    params = {
        "out": "json",
        "lim": 500,
        "userid": user_id,
        "of": "n-t-s-w-k-gl-l-nt-e-a-f-gf-ga-ti-dp-wp-mp-qp-yp-imp-r-ah-ka-g",
    }
    
    try:
        response = requests.get(NAROU_API_URL, params=params)
        response.raise_for_status()
        data = response.json()
        
        if isinstance(data, list) and len(data) > 1 and isinstance(data[1], dict):
            return data[1:]
        else:
            return []
            
    except requests.exceptions.RequestException as e:
        # print(f"\nAPIリクエスト中にエラーが発生しました: {e}")
        return None # エラー発生時はNoneを返す
    except ValueError as e:
        # JSON解析エラーはリトライしても成功しない可能性が高いため、エラーとして記録
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(ERROR_LOG_PATH, 'a', encoding='utf-8') as f:
            f.write(f"{timestamp}, JSON_PARSE_ERROR_UID: {user_id}, Details: {e}\n")
        return None

In [5]:
# --- 1回の実行で取得を試みるユーザ数を設定 ---
NUM_USERS_TO_TRY = 150000

# --- リトライ設定 ---
MAX_RETRIES = 3  # 最大リトライ回数
RETRY_WAIT_SECONDS = 5 # リトライ時の待機時間（秒）

# --- 準備 ---
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)
    print(f"'{DATASET_DIR}' ディレクトリを作成しました。")

if os.path.exists(PROGRESS_FILE_PATH):
    with open(PROGRESS_FILE_PATH, 'r') as f:
        try:
            start_uid = int(f.read().strip()) + 1
        except ValueError:
            start_uid = 1
else:
    start_uid = 1
print(f"UID {start_uid} から収集を開始します。")

if os.path.exists(RAW_DATA_PATH):
    print(f"既存の生データ'{RAW_DATA_PATH}'を読み込みます。")
    try:
        existing_raw_df = pd.read_csv(RAW_DATA_PATH)
        collected_ncodes = set(existing_raw_df['ncode'])
        print(f"現在の収集済み作品数: {len(collected_ncodes)}件")
    except (pd.errors.EmptyDataError, KeyError):
        existing_raw_df = pd.DataFrame()
        collected_ncodes = set()
else:
    existing_raw_df = pd.DataFrame()
    collected_ncodes = set()

# --- データ収集実行 ---
print("\n--- データ収集を開始します ---")
newly_collected_novels = []
last_checked_uid = start_uid - 1

for i in range(NUM_USERS_TO_TRY):
    current_uid = start_uid + i
    if current_uid > MAX_USER_ID:
        print("\n最大ユーザIDに達したため、収集を終了します。")
        break
    
    retry_count = 0
    while retry_count < MAX_RETRIES:
        # 実行状況を表示
        progress_message = f"\r試行 {i+1}/{NUM_USERS_TO_TRY} | User ID: {current_uid:<8} | 新規収集: {len(newly_collected_novels):<5}件"
        if retry_count > 0:
            progress_message += f" (リトライ {retry_count}/{MAX_RETRIES})"
        print(progress_message, end="", flush=True)

        # データ取得を試行
        novels = get_narou_data_by_userid(current_uid)
        
        # --- 成功した場合 ---
        if novels is not None:
            if novels: # 空リストでない場合のみ処理
                for novel in novels:
                    ncode = novel.get('ncode')
                    if ncode and ncode not in collected_ncodes:
                        newly_collected_novels.append(novel)
                        collected_ncodes.add(ncode)
            break # リトライ処理を抜けて次のUIDへ
        
        # --- 失敗した場合 (novels is None) ---
        retry_count += 1
        if retry_count < MAX_RETRIES:
            # 次のリトライまで待機
            time.sleep(RETRY_WAIT_SECONDS)
        else:
            # 最大リトライ回数に達した場合
            print(f"\nUser ID: {current_uid} の取得に失敗しました。エラーを記録して次に進みます。")
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            with open(ERROR_LOG_PATH, 'a', encoding='utf-8') as f:
                f.write(f"{timestamp}, FAILED_UID_AFTER_RETRIES: {current_uid}\n")
    
    last_checked_uid = current_uid
    # 通常のリクエスト間隔の待機
    time.sleep(REQUEST_INTERVAL)

# --- 進捗の保存 ---
with open(PROGRESS_FILE_PATH, 'w') as f:
    f.write(str(last_checked_uid))
print(f"\n\nデータ収集完了。最後にチェックしたUID: {last_checked_uid} を記録しました。")

# --- 収集データの保存と整形 ---
if not newly_collected_novels:
    print("今回は新しい小説を収集できませんでした。")
else:
    print(f"今回新たに {len(newly_collected_novels)}件 の小説を収集しました。")
    new_df = pd.DataFrame(newly_collected_novels)
    
    combined_raw_df = pd.concat([existing_raw_df, new_df], ignore_index=True)
    combined_raw_df.to_csv(RAW_DATA_PATH, index=False, encoding='utf-8-sig')
    print(f"生データを'{RAW_DATA_PATH}'に保存しました。現在の生データ合計: {len(combined_raw_df)}件")

    print("\n--- データ整形処理を開始します ---")
    df = combined_raw_df.copy()

    if 'noveltype' in df.columns:
        num_before = len(df)
        df = df[df['noveltype'] == 1].copy()
        print(f"1. 短編小説を除外しました ({num_before} -> {len(df)}件)")
    
    df.rename(columns={
        'ncode': 'Nコード', 'title': 'タイトル', 'story': 'あらすじ', 'writer': '作者', 'keyword': 'キーワード',
        'general_firstup': '初回投稿日', 'general_lastup': '最終更新日', 'noveltype': '小説タイプ', 'end': '完結フラグ',
        'general_all_no': '投稿話数', 'length': '文字数', 'time': '読了時間', 'kaiwaritu': '会話率',
        'fav_novel_cnt': 'ブックマーク数', 'impression_cnt': '感想数', 'review_cnt': 'レビュー数',
        'all_point': '総合評価ポイント', 'all_hyoka_cnt': '評価者数', 'daily_point': '日間ポイント',
        'weekly_point': '週間ポイント', 'monthly_point': '月間ポイント', 'quarter_point': '四半期ポイント',
        'yearly_point': '年間ポイント', 'genre': '作品ジャンル'
    }, inplace=True)
    
    print("2. is_eternalラベル（エタり判定）を計算します...")
    df['完結フラグ'] = pd.to_numeric(df['完結フラグ'], errors='coerce').fillna(0).astype(int)
    df['最終更新日_dt'] = pd.to_datetime(df['最終更新日'])
    
    one_year_ago = datetime.now() - timedelta(days=365)
    is_eternal_condition = (df['完結フラグ'] == 0) & (df['最終更新日_dt'] < one_year_ago)
    df['is_eternal'] = is_eternal_condition.astype(int)

    print("3. アクティブな連載作品（分析対象外）を除外します...")
    num_before = len(df)
    condition_to_exclude = (df['完結フラグ'] == 0) & (df['is_eternal'] == 0)
    df = df[~condition_to_exclude].copy()
    print(f"   除外により {num_before - len(df)}件 を削除しました。")

    df.drop(columns=['最終更新日_dt'], inplace=True, errors='ignore')
    
    df.to_csv(PROCESSED_DATA_PATH, index=False, encoding='utf-8-sig')
    print(f"\n処理済みの最終的なコーパスを'{PROCESSED_DATA_PATH}'に保存しました。")
    print(f"最終的な作品数: {len(df)}件")
    if 'is_eternal' in df.columns:
        print("\n--- 最終コーパスのデータ内訳 ---")
        print(df['is_eternal'].value_counts())

UID 2700001 から収集を開始します。
既存の生データ'dataset\narou_dataset_raw.csv'を読み込みます。
現在の収集済み作品数: 1025619件

--- データ収集を開始します ---
試行 114253/150000 | User ID: 2814253  | 新規収集: 26006件
最大ユーザIDに達したため、収集を終了します。


データ収集完了。最後にチェックしたUID: 2814253 を記録しました。
今回新たに 26006件 の小説を収集しました。
生データを'dataset\narou_dataset_raw.csv'に保存しました。現在の生データ合計: 1051625件

--- データ整形処理を開始します ---
1. 短編小説を除外しました (1051625 -> 555112件)
2. is_eternalラベル（エタり判定）を計算します...
3. アクティブな連載作品（分析対象外）を除外します...
   除外により 17283件 を削除しました。

処理済みの最終的なコーパスを'dataset\narou_dataset.csv'に保存しました。
最終的な作品数: 537829件

--- 最終コーパスのデータ内訳 ---
is_eternal
0    391586
1    146243
Name: count, dtype: int64
